# E002 — What Actually Wins? Ladder Forensics

**Input checklist**
- Required artifact from E001: `turns.parquet`.
- Accelerator: **None / CPU**.
- Internet: **OFF**.

**Goal:** extend ladder analysis around pairwise win probability and test whether strong agents are approximately open-loop.

In [ ]:
from pathlib import Path
import os,sys,json
SUITE_CANDIDATES=[Path('/kaggle/input/kaggriculture-v2-suite'),Path('/kaggle/input/kaggriculture-v2-suite/kaggriculture_v2_suite'),Path.cwd().parent,Path.cwd()]
ROOT=next((p for p in SUITE_CANDIDATES if (p/'src'/'kagv2').exists()),None)
if ROOT is None: raise FileNotFoundError('Attach/upload kaggriculture_v2_suite as a Kaggle Dataset, or run this notebook inside the repo.')
sys.path.insert(0,str(ROOT)); WORK=Path('/kaggle/working/kagv2') if Path('/kaggle/working').exists() else ROOT/'artifacts'; WORK.mkdir(parents=True,exist_ok=True)
print('ROOT=',ROOT,'WORK=',WORK)

In [ ]:
import pandas as pd,numpy as np,matplotlib.pyplot as plt
from src.kagv2.ladder import open_loop_report,deduplicated_matchups,bradley_terry
turns=pd.read_parquet(WORK/'turns.parquet')
match=deduplicated_matchups(turns);print('matches',len(match));display(match.head())
bt=bradley_terry(match);bt.to_csv(WORK/'bt_strength.csv',index=False);display(bt.head(30))

In [ ]:
ol=open_loop_report(turns,min_episodes=3);ol.to_csv(WORK/'open_loop_report.csv',index=False);display(ol.head(40))
if len(ol)>3:
    plt.figure(figsize=(7,5));plt.scatter(ol.open_loop_score,ol.mean_reward);plt.xlabel('open-loop score (1 - action entropy)');plt.ylabel('mean final coins');plt.title('Does open-loop behavior correlate with raw economy?');plt.show()

In [ ]:
if not match.empty:
    match['margin']=(match.reward_a-match.reward_b).abs();print(match.groupby('y').margin.describe())
wins=turns[turns.win_target==1]
for d in [3,7,11,15,20,25,29]:
    x=wins[(wins.day==d)&(wins.hour==0)]
    if len(x):
        cols=['crop_WHEAT','crop_STRAWBERRY','crop_MELON','animal_COW','animal_SHEEP','quadrants','hands']
        print('day',d);display(x[cols].describe().loc[['50%','mean']])

### Interpretation rule
If the best actors have high open-loop scores across many opponents, prioritize macro imitation + CEM. If open-loopness falls as strength rises, increase the weight of opponent-conditioned models.